# Prepare local source metadata

Licensed indexes → validated paths and clocks → manifests in `data/`.
Audio stays under one explicit `AUDIO_ROOT`. No source download runs here.
Loop use is pending until its release terms are recorded; raw recording text is not aligned training data.

In [ ]:
%pip install -q -e '/kaggle/working/shono[audio]'

In [ ]:
from pathlib import Path

from shono.data import (
    LicenseRegistry, from_common_voice_tsv, from_fleurs_tsv, from_slr53_tsv,
    from_mucs_kaldi, from_loop_jsonl, audit_leakage, validate_manifest_audio,
)
from shono.output_paths import validate_output_destinations

REPO = Path('/kaggle/working/shono')
DATA = REPO / 'data'
AUDIO_ROOT = Path('/kaggle/input')
CV_ROOT = AUDIO_ROOT / 'common-voice-bn'
FLEURS_ROOT = AUDIO_ROOT / 'fleurs-bn'
SLR53_ROOT = AUDIO_ROOT / 'openslr53'
MUCS_ROOT = AUDIO_ROOT / 'mucs-slr104'
LOOP_ROOT = AUDIO_ROOT / 'bengali-loop'
registry = LicenseRegistry.load(DATA / 'licenses.json')
HASH_AUDIO = False  # True computes complete-file SHA256; False leaves coverage unknown.
manifests = []
source_indexes = []

def prefix(root):
    return root.relative_to(AUDIO_ROOT).as_posix()

def collect(manifest):
    manifest.validate_against(registry)
    validate_manifest_audio(manifest, AUDIO_ROOT)
    manifests.append(manifest)
    print(f'{manifest.name}: {len(manifest.segments)} segments, {manifest.total_hours():.2f} h')

## Common Voice and FLEURS

The Common Voice speaker ID remains the bootstrap block, separate from its clip path.
The Common Voice development split is created for training evaluation.

In [ ]:
if (CV_ROOT / 'train.tsv').is_file():
    duration_index = CV_ROOT / 'clip_durations.tsv'
    for split in ('train', 'dev', 'test'):
        index = CV_ROOT / f'{split}.tsv'
        source_indexes.append(index)
        collect(from_common_voice_tsv(
            index, split=split, name=f'cv-bn-{split}', version='cv-corpus-26.0',
            audio_root=AUDIO_ROOT, audio_subdir=f'{prefix(CV_ROOT)}/clips',
            durations_tsv=duration_index if duration_index.is_file() else None,
            hash_audio=HASH_AUDIO,
        ))
    if duration_index.is_file(): source_indexes.append(duration_index)
else:
    print('pending: Common Voice local indexes missing')

if (FLEURS_ROOT / 'test.tsv').is_file():
    source_indexes.append(FLEURS_ROOT / 'test.tsv')
    collect(from_fleurs_tsv(
        FLEURS_ROOT / 'test.tsv', split='test', name='fleurs-bn-test',
        audio_root=AUDIO_ROOT, audio_subdir=f'{prefix(FLEURS_ROOT)}/audio',
        hash_audio=HASH_AUDIO,
    ))
else:
    print('pending: FLEURS local index missing')

## SLR53 and MUCS

SLR53 publishes `utt_spk_text.tsv`: FileID, anonymized UserID, transcription.
WAV names are discovered under the selected local dataset root, without guessing archive folders.
MUCS uses explicit Kaldi joins; shell commands in `wav.scp` are rejected.

In [ ]:
if (SLR53_ROOT / 'utt_spk_text.tsv').is_file():
    source_indexes.append(SLR53_ROOT / 'utt_spk_text.tsv')
    collect(from_slr53_tsv(
        SLR53_ROOT / 'utt_spk_text.tsv', name='slr53-train', split='train',
        audio_root=AUDIO_ROOT, dataset_root=SLR53_ROOT, hash_audio=HASH_AUDIO,
    ))
else:
    print('pending: SLR53 local published index missing')

for split in ('train', 'test'):
    local = MUCS_ROOT / split  # Set this to the downloaded Kaldi index directory.
    if not (local / 'text').is_file():
        print(f'pending: MUCS {split} local indexes missing')
        continue
    source_indexes.extend([local / 'text', local / 'wav.scp'])
    segments = local / 'segments'
    speakers = local / 'utt2spk'
    for index in (segments, speakers):
        if index.is_file(): source_indexes.append(index)
    collect(from_mucs_kaldi(
        local / 'text', local / 'wav.scp', name=f'mucs-cs-{split}', split=split,
        audio_root=AUDIO_ROOT, path_root=local,
        segments_path=segments if segments.is_file() else None,
        utt2spk_path=speakers if speakers.is_file() else None, hash_audio=HASH_AUDIO,
    ))

## Loop recording metadata

Set the two JSONL filenames to the actual published or locally selected recording indexes.
No alignment or release permission is inferred from a paper citation.

In [ ]:
LOOP_INDEXES = {
    'bengali-loop-sample': LOOP_ROOT / 'asr_sample.jsonl',
    'bengali-loop-test': LOOP_ROOT / 'asr_test.jsonl',
}
if registry.require('bengali_loop').status == 'excluded':
    print('pending: Bengali-Loop release permissions are unconfirmed')
else:
    for name, index in LOOP_INDEXES.items():
        if not index.is_file():
            print(f'pending: {name} local recording index missing')
            continue
        source_indexes.append(index)
        collect(from_loop_jsonl(index, name=name, version='operator-recorded-release',
                                audio_root=AUDIO_ROOT, audio_prefix=prefix(LOOP_ROOT)))

## Inspect overlap, then write the validated metadata

This preparation summary is advisory. Training and scored fine-tuned reports run the
mandatory guard again and require exact normalized text keys with written review reasons.
All output paths are checked before the first manifest write.

In [ ]:
trains = [m for m in manifests if m.split == 'train']
for evaluation in (m for m in manifests if m.split in ('dev', 'test')):
    if trains:
        print(audit_leakage(trains, evaluation, audio_root=AUDIO_ROOT).summary())

paths = [DATA / f'{m.name}.manifest.jsonl' for m in manifests]
protected = [*source_indexes, DATA / 'licenses.json', REPO / 'LICENSE']
protected += [path for m in manifests for path in validate_manifest_audio(m, AUDIO_ROOT)]
validate_output_destinations(paths, protected_paths=protected, output_root=DATA)
DATA.mkdir(parents=True, exist_ok=True)
for manifest, path in zip(manifests, paths, strict=True):
    manifest.to_jsonl(path)
print(f'wrote {len(manifests)} source manifests; no audio was copied')